# 07 — v1.3 Benchmark Suite and Dashboard Artifacts

**Goal:** Run the TGraphX v1.3 smoke benchmark suite, inspect the JSON
results, and understand how to point the local dashboard to the artifact
directory.

**TGraphX subsystem:** `benchmarks/run_v13_benchmark_suite.py`

**Data:** Synthetic — no download.

**Runtime:** < 120 seconds on CPU.

**Important note:** These are **smoke benchmarks** — tiny synthetic data.
They are NOT competitive throughput claims against PyG, DGL, PyKEEN, or SB3.

In [ ]:
import subprocess, json, sys
result = subprocess.run(
    [sys.executable, "benchmarks/run_v13_benchmark_suite.py", "--small", "--json"],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print("STDERR:", result.stderr[:500])
else:
    data = json.loads(result.stdout)
    print(f"Suite: {data['suite']}")
    print(f"Version: {data['package_version']}  Device: {data['device']}")
    print()
    for row in data['benchmarks']:
        status = row['status']
        rt = f"{row['runtime_s']:.3f}s" if row['runtime_s'] else "failed"
        print(f"  {row['name']:<35} {status:<7} {rt}")

## 2. Inspect Individual Benchmark Metrics

In [ ]:
# Show metrics from successful rows.
for row in data['benchmarks']:
    if row['status'] == 'ok' and row['metrics']:
        print(f"\n{row['name']}:")
        for k, v in row['metrics'].items():
            print(f"  {k}: {v}")

## 3. Write Dashboard-Compatible Output

In [ ]:
import tempfile, pathlib
with tempfile.TemporaryDirectory() as d:
    # Write the benchmark JSON to a directory.
    out = pathlib.Path(d) / "benchmark_results.json"
    out.write_text(json.dumps(data, indent=2))
    print("Written:", out)
    print("\nTo view in the TGraphX dashboard:")
    print(f"  tgraphx-dashboard --logdir {d}")
    print("\n(The dashboard reads benchmark_results.json and similar files.)")

## 4. Benchmark Scope

| Type | What it measures | What it does NOT claim |
|---|---|---|
| Smoke benchmark | Correctness + basic runtime | Competitive throughput vs PyG/DGL |
| Performance benchmark | Local machine timing | Cross-machine reproducibility |
| Reference benchmark | Algorithm correctness | SOTA performance |

The honest scope is in `docs/benchmark_report.md`.

## 5. Next Steps
- **Full suite:** `benchmarks/run_v13_benchmark_suite.py`
- **Report:** `docs/benchmark_report.md`
- **Dashboard:** `tgraphx-dashboard --logdir <dir>`